# AdaptiveMath-AI and matched ablations

Executable scientific definitions and computed results are presented below. Data, fitted models, tables and figures are stored in the corresponding standard project directories. Earlier experiments are preserved separately in `Data/legacy/Notebooks/` and are not mixed with the current results.

In [1]:
from pathlib import Path
import sys, types, hashlib, importlib.abc, importlib.util
import nbformat
import pandas as pd
from IPython.core.magic import register_cell_magic
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'Notebooks').is_dir() and (p/'Data').is_dir())
MODULE_NOTEBOOKS={'revision_data': '02_data_preprocessing_feature_engineering_and_splits.ipynb', 'revision_models': '03_baseline_models.ipynb', 'revision_sensitivity': '02_data_preprocessing_feature_engineering_and_splits.ipynb', 'revision_neural': '04_sota_models.ipynb', 'revision_evaluation': '06_final_validation_tables_figures_and_reports.ipynb', 'revision_supplemental': '06_final_validation_tables_figures_and_reports.ipynb', 'revision_provenance': '05_proposed_hybrid_model_and_ablations.ipynb', 'revision_validation': '02_data_preprocessing_feature_engineering_and_splits.ipynb', 'revision_closure': '06_final_validation_tables_figures_and_reports.ipynb', 'revision_status': '06_final_validation_tables_figures_and_reports.ipynb'}

class NotebookSourceLoader(importlib.abc.Loader):
    def create_module(self,spec):return None
    def exec_module(self,module):
        path=ROOT/'Notebooks'/MODULE_NOTEBOOKS[module.__name__]
        notebook=nbformat.read(path,4)
        cell=next(c for c in notebook.cells if c.metadata.get('research_module')==module.__name__)
        source=cell.source.split('\n',1)[1]
        module.__file__=str(path)
        module.__notebook_source_sha256__=hashlib.sha256(source.encode()).hexdigest()
        exec(compile(source,str(path)+'#'+cell.id,'exec'),module.__dict__)

class NotebookSourceFinder(importlib.abc.MetaPathFinder):
    def find_spec(self,fullname,path=None,target=None):
        if fullname in MODULE_NOTEBOOKS:
            return importlib.util.spec_from_loader(fullname,NotebookSourceLoader())
sys.meta_path=[f for f in sys.meta_path if type(f).__name__!='NotebookSourceFinder']
sys.meta_path.insert(0,NotebookSourceFinder())

@register_cell_magic
def research_module(line,source):
    """Publish the visible functions for reuse by other notebooks; no hidden helper scripts."""
    name=line.strip();digest=hashlib.sha256(source.rstrip('\n').encode()).hexdigest()
    existing=sys.modules.get(name)
    if existing is not None and existing.__notebook_source_sha256__==digest:return
    module=types.ModuleType(name);module.__file__=str(ROOT/'Notebooks'/MODULE_NOTEBOOKS[name])
    module.__notebook_source_sha256__=digest;sys.modules[name]=module
    exec(compile(source.rstrip('\n'),module.__file__,'exec'),module.__dict__)

@register_cell_magic
def legacy_snapshot(line,cell):
    """Archived analysis is preserved but is not part of the current execution."""
    return None

import revision_data as rd
artifact_path=rd.artifact_path
import matplotlib as mpl
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('png')
mpl.rcParams.update({'figure.dpi':350,'savefig.dpi':350})
pd.set_option('display.max_rows',None)
pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth',100)

### Provenance — executable definitions

The functions below are the source used by this notebook and reused by the other notebooks. Defining them does not repeat model fitting.

In [2]:
%%research_module revision_provenance
"""Frozen-component reconstruction and compact prediction provenance."""
from revision_data import artifact_path, code_digest, source_matches, artifact_matches, fit_contract_matches
import json
import joblib
import numpy as np
import pandas as pd
from scipy.special import expit,logit
from threadpoolctl import threadpool_limits
from revision_data import V,KEYS,protocol,digest,save_json,table,object_hash
from revision_models import predict,calibrate,hierarchy_apply,reuse_converged_helper,safe_p

@threadpool_limits.wrap(limits=4)
def component_provenance():
    features=pd.read_parquet(artifact_path('Data/features_primary.parquet')).set_index('AnswerId',drop=False)
    records=[];routes=[];fits=[];helper=reuse_converged_helper()
    for path in sorted((artifact_path('Data')).glob('predictions_seed*_fold*.parquet')):
        pred=pd.read_parquet(path);seed=int(pred.seed.iloc[0]);fold=int(pred.fold.iloc[0])
        ids=pd.read_parquet(artifact_path('Data')/f'fitting_ids_seed{seed}_fold{fold}.parquet')
        ap=artifact_path('Models')/f'AdaptiveMath_AI_seed{seed}_fold{fold}.joblib';obj=joblib.load(ap)
        target=features.loc[pred.loc[pred.model.eq('AdaptiveMath-AI'),'AnswerId']].reset_index(drop=True)
        anchor=predict(obj['anchor'],target);memids=ids[(ids.model=='AdaptiveMath-AI')&(ids.component=='memory')].AnswerId
        mem=features.loc[memids].reset_index(drop=True);mp=predict(obj['anchor'],mem)
        corrections,iterations=helper(mem.rename(columns={'QuestionId':'question_id'}),mem.IsCorrect.to_numpy(),mp,shrinkage=obj['selected']['question'][0])
        cp=artifact_path('Models')/f'converged_parameters_seed{seed}_fold{fold}.joblib'
        joblib.dump({'delta':corrections,'iterations':iterations,'lambda':obj['selected']['question'][0],'cap':obj['selected']['question'][1],'parent_model_hash':digest(ap),'memory_ids_hash':object_hash(memids.tolist())},cp)
        for name,g in pred.groupby('model',sort=False):
            recipe={'variant':name,'parent_model_hash':g.model_hash.iloc[0],'calibration':'none','memory_variant':'none'}
            route=None
            if name in ['HGB80','HGB100']:
                own=joblib.load(artifact_path('Models')/f'{name}_seed{seed}_fold{fold}.joblib')
                p=calibrate(own['calibrator'],predict(own['anchor'],target));recipe['calibration']=joblib.hash(own['calibrator'])
            elif name=='HGB80_anchor':p=anchor
            elif name=='Converged_item_intercept':
                delta=target.QuestionId.astype(str).map(corrections).fillna(0).clip(-obj['selected']['question'][1],obj['selected']['question'][1]).to_numpy()
                p=safe_p(expit(logit(anchor)+delta));recipe['converged_parameters_hash']=digest(cp)
            else:
                variant={'Plus_question_memory':'question_only','Ablation_no_question':'no_question','Ablation_no_subject':'no_subject'}.get(name,'full')
                p,route=hierarchy_apply(target,anchor,obj['memory'],variant);recipe['memory_variant']=variant
                if name in ['AdaptiveMath-AI','Ablation_no_question','Ablation_no_subject']:
                    p=calibrate(obj['calibrator'],p);recipe['calibration']=joblib.hash(obj['calibrator'])
            # Target order is fixed in all variant frames; check IDs explicitly.
            assert np.array_equal(g.AnswerId.to_numpy(),target.AnswerId.to_numpy())
            np.testing.assert_allclose(p,g.p.to_numpy(),rtol=0,atol=1e-12)
            rid=object_hash(recipe)
            records.append({'prediction_file':str(path.relative_to(V)),'file_sha256':digest(path),'model':name,'seed':seed,'fold':fold,'N':len(g),'prediction_kind':g.prediction_kind.iloc[0],'model_hash_semantics':'frozen object' if name in ['AdaptiveMath-AI','HGB80','HGB100'] else 'parent frozen object plus explicit reconstruction recipe','recipe_hash':rid,'recipe':json.dumps(recipe,sort_keys=True),'reloaded_prediction_max_error':float(np.max(np.abs(p-g.p.to_numpy())))})
            if route is not None:
                for route_name,r in route.groupby('route'):
                    routes.append({'model':name,'seed':seed,'fold':fold,'route':route_name,'N':len(r),'route_fraction':len(r)/len(route),'support_median':r.support_n.median(),'support_min':r.support_n.min(),'support_max':r.support_n.max(),'clipped_fraction':r.clipped.mean(),'mean_abs_correction':r.correction.abs().mean(),'shrinkage_lambda':r.shrinkage_lambda.iloc[0]})
        for (name,component),g in ids.groupby(['model','component']):
            assert g.DateAnswered.max()<target.DateAnswered.min()
            fits.append({'model':name,'seed':seed,'fold':fold,'component':component,'fit_rows':len(g),'fit_start':g.DateAnswered.min(),'fit_end':g.DateAnswered.max(),'evaluation_start':target.DateAnswered.min(),'exact_ids_path':str((artifact_path('Data')/f'fitting_ids_seed{seed}_fold{fold}.parquet').relative_to(V))})
        print(f'Frozen component reconstruction verified: seed={seed}, fold={fold}',flush=True)
    table('prediction_component_provenance.csv',pd.DataFrame(records),'NB06')
    table('component_fitting_provenance.csv',pd.DataFrame(fits),'NB06')
    table('route_support_diagnostics.csv',pd.DataFrame(routes),'NB06')
    return pd.DataFrame(records).groupby('model').agg(folds=('fold','size'),maximum_reload_error=('reloaded_prediction_max_error','max')).reset_index()

In [3]:
import revision_models as rm
display(rm.primary_experiment())

Validated exact cache seed=20260713 fold=0


Validated exact cache seed=20260713 fold=1


Validated exact cache seed=20260713 fold=2


Validated exact cache seed=20260713 fold=3


Validated exact cache seed=20260713 fold=4


...

Validated exact cache seed=20260722 fold=1


Validated exact cache seed=20260722 fold=2


Validated exact cache seed=20260722 fold=3


Validated exact cache seed=20260722 fold=4


ROC_AUC            Log_Loss               Brier  \
                              mean       std      mean       std      mean   
model                                                                        
Ablation_no_question      0.755992  0.005780  0.554532  0.018488  0.188533   
Ablation_no_subject       0.760359  0.006207  0.551156  0.018857  0.187123   
AdaptiveMath-AI           0.760329  0.006287  0.551143  0.018916  0.187116   
Converged_item_intercept  0.760350  0.006307  0.551021  0.018931  0.187070   
HGB100                    0.756085  0.006382  0.554315  0.018805  0.188505   
HGB80                     0.756219  0.006132  0.554119  0.018691  0.188421   
HGB80_anchor              0.755688  0.005833  0.554549  0.018553  0.188544   
No_calibration            0.760329  0.006287  0.551080  0.018937  0.187093   
Plus_hierarchy            0.760329  0.006287  0.551080  0.018937  0.187093   
Plus_question_memory      0.760353  0.006303  0.551017  0.018926  0.187068   

                                       ECE15            
                               std      mean       std  
model                                                   
Ablation_no_question      0.007800  0.014109  0.003828  
Ablation_no_subject       0.007901  0.014738  0.003386  
AdaptiveMath-AI           0.007928  0.014003  0.003187  
Converged_item_intercept  0.007937  0.012933  0.003334  
HGB100                    0.007954  0.014864  0.004783  
HGB80                     0.007917  0.013414  0.003878  
HGB80_anchor              0.007827  0.011983  0.003887  
No_calibration            0.007935  0.013628  0.003448  
Plus_hierarchy            0.007935  0.013628  0.003448  
Plus_question_memory      0.007935  0.012901  0.003335